# Predicting survival on the Titanic

In [1]:
import pandas as pd  
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

#Load the dataset 
test_data = pd.read_csv("test.csv")
train_data = pd.read_csv("train.csv")

## Preprocessing Testing Data 

In [2]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [3]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Load the dataset 
test_data = pd.read_csv("test.csv")
train_data = pd.read_csv("train.csv")

# Preprocessing Testing Data
# Fill missing Age values with the mean age from the training data
test_fill_in = train_data['Age'].mean()
test_data['Age'] = test_data['Age'].fillna(test_fill_in)

# Updated Optimization for Age Categories
new_age_bins = [0, 12, 18, 30, 60, 80, 100]
new_age_labels = [0, 1, 2, 3, 4, 5]
test_data['AgeCategories'] = pd.cut(test_data['Age'].copy(), bins = new_age_bins, labels = new_age_labels)
test_data['AgeCategories'] = test_data['AgeCategories'].astype(int)

# Updated Optimization for Family sizes and checking to see if bigger fams survive more
test_data['FamilySize'] = (test_data['SibSp'] + (test_data['Parch'] + 1))

#Updated Optimization seeing if solo chances of survival 
test_data['Solo'] = (test_data['FamilySize'] == 1).astype(int)

#Adding and seeeing fare per person
test_data['FarePer'] = test_data['Fare'] / test_data['FamilySize']

# Fill missing Fare values with the median fare from the training data
test_fill_in_fare = train_data['Fare'].median()
test_data['Fare'] = test_data['Fare'].fillna(test_fill_in_fare)

# Fill missing Embarked values with 'S'
test_data['Embarked'] = test_data['Embarked'].fillna('S')

# One-hot encode Embarked
test_data = pd.get_dummies(test_data, columns=['Embarked'], prefix='Embarked')

# Ensure consistency with training columns
for col in ['Embarked_C', 'Embarked_Q', 'Embarked_S']:
    if col not in test_data.columns:
        test_data[col] = 0

# Map Sex to numerical values
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})

# Dropping unnecessary columns + handling missing values
test_cleaned = test_data.drop(['Name', 'Cabin', 'Ticket'], axis=1, errors='ignore')

In [5]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,AgeCategories,FamilySize,Solo,FarePer,Embarked_C,Embarked_Q,Embarked_S
0,892,3,"Kelly, Mr. James",0,34.5,0,0,330911,7.8292,NaN,3,1,1,7.829200,False,True,False
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",1,47.0,1,0,363272,7.0000,NaN,3,2,0,3.500000,False,False,True
2,894,2,"Myles, Mr. Thomas Francis",0,62.0,0,0,240276,9.6875,NaN,4,1,1,9.687500,False,True,False
3,895,3,"Wirz, Mr. Albert",0,27.0,0,0,315154,8.6625,NaN,2,1,1,8.662500,False,False,True
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1,22.0,1,1,3101298,12.2875,NaN,2,3,0,4.095833,False,False,True


In [6]:

test_cleaned.head()

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,AgeCategories,FamilySize,Solo,FarePer,Embarked_C,Embarked_Q,Embarked_S
0,892,3,0,34.5,0,0,7.8292,3,1,1,7.829200,False,True,False
1,893,3,1,47.0,1,0,7.0000,3,2,0,3.500000,False,False,True
2,894,2,0,62.0,0,0,9.6875,4,1,1,9.687500,False,True,False
3,895,3,0,27.0,0,0,8.6625,2,1,1,8.662500,False,False,True
4,896,3,1,22.0,1,1,12.2875,2,3,0,4.095833,False,False,True


## Preprocessing Training Data 

In [7]:
# fill missing Age values with mean
train_fill_in = train_data['Age'].mean()
train_data['Age'] = train_data['Age'].fillna(train_fill_in)

# Updated Optimization for Age
new_age_bins = [0, 12, 18, 30, 60, 80, 100]
new_age_labels = [0, 1, 2, 3, 4, 5]
train_data['AgeCategories'] = pd.cut(train_data['Age'], bins=new_age_bins, labels=new_age_labels)
train_data['AgeCategories'] = train_data['AgeCategories'].astype(int)

# Updated Optimization for Family sizes and checking to see if bigger fams survive more
train_data['FamilySize'] = (train_data['SibSp'] + (train_data['Parch'] + 1))

# Updated Optimization seeing if solo chances of survival 
train_data['Solo'] = (train_data['FamilySize'] == 1).astype(int)
#Adding and seeeing fare per person
train_data['FarePer'] = train_data['Fare'] / train_data['FamilySize']

# Fill missing Embarked values with most common port ('S')
train_data['Embarked'] = train_data['Embarked'].fillna('S')

# Encode Sex (male=0, female=1)
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1}).fillna(0)

# fill missing Fare values if needed
train_fill_in_fare = train_data['Fare'].median()
train_data['Fare'] = train_data['Fare'].fillna(train_fill_in_fare)

# One-hot encode Embarked
train_data = pd.get_dummies(train_data, columns=['Embarked'], prefix='Embarked')

# Ensure all three dummy columns exist
for col in ['Embarked_C', 'Embarked_Q', 'Embarked_S']:
    if col not in train_data.columns:
        train_data[col] = 0

In [8]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,AgeCategories,FamilySize,Solo,FarePer,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,2,2,0,3.62500,False,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,3,2,0,35.64165,True,False,False
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,2,1,1,7.92500,False,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,3,2,0,26.55000,False,False,True
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,3,1,1,8.05000,False,False,True


In [9]:
print(train_data.isnull().sum())
print(test_cleaned.isnull().sum())

PassengerId        0
Survived           0
Pclass             0
Name               0
Sex                0
Age                0
SibSp              0
Parch              0
Ticket             0
Fare               0
Cabin            687
AgeCategories      0
FamilySize         0
Solo               0
FarePer            0
Embarked_C         0
Embarked_Q         0
Embarked_S         0
dtype: int64
PassengerId      0
Pclass           0
Sex              0
Age              0
SibSp            0
Parch            0
Fare             0
AgeCategories    0
FamilySize       0
Solo             0
FarePer          1
Embarked_C       0
Embarked_Q       0
Embarked_S       0
dtype: int64


## Building the Models 

In [10]:
#Binary Classification Model Implementation
#Referencing: MN Lab 4 Neural Network Implementation 

#splitting into x(features) and y(target) + training and testing sets
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'AgeCategories', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'FamilySize', 'Solo', 'FarePer']
X = train_data[features] #every col but 'survived'
y = train_data ['Survived']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2 ,random_state=42) #42 because that's what we use in class

In [11]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Load the dataset 
test_data = pd.read_csv("test.csv")
train_data = pd.read_csv("train.csv")

# Preprocessing Testing Data
# Fill missing Age values with the mean age from the training data
test_fill_in = train_data['Age'].mean()
test_data['Age'] = test_data['Age'].fillna(test_fill_in)

# Updated Optimization for Age Categories
new_age_bins = [0, 12, 18, 30, 60, 80, 100]
new_age_labels = [0, 1, 2, 3, 4, 5]
test_data['AgeCategories'] = pd.cut(test_data['Age'].copy(), bins = new_age_bins, labels = new_age_labels)
test_data['AgeCategories'] = test_data['AgeCategories'].astype(int)

# Updated Optimization for Family sizes and checking to see if bigger fams survive more
test_data['FamilySize'] = (test_data['SibSp'] + (test_data['Parch'] + 1))

#Updated Optimization seeing if solo chances of survival 
test_data['Solo'] = (test_data['FamilySize'] == 1).astype(int)

#Adding and seeeing fare per person
test_data['FarePer'] = test_data['Fare'] / test_data['FamilySize']

# Fill missing Fare values with the median fare from the training data
test_fill_in_fare = train_data['Fare'].median()
test_data['Fare'] = test_data['Fare'].fillna(test_fill_in_fare)

# Fill missing Embarked values with 'S'
test_data['Embarked'] = test_data['Embarked'].fillna('S')

# One-hot encode Embarked
test_data = pd.get_dummies(test_data, columns=['Embarked'], prefix='Embarked')

# Ensure consistency with training columns
for col in ['Embarked_C', 'Embarked_Q', 'Embarked_S']:
    if col not in test_data.columns:
        test_data[col] = 0

# Map Sex to numerical values
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})

# Dropping unnecessary columns + handling missing values
test_cleaned = test_data.drop(['Name', 'Cabin', 'Ticket'], axis=1, errors='ignore')

# Preprocessing Training Data
# fill missing Age values with mean
train_fill_in = train_data['Age'].mean()
train_data['Age'] = train_data['Age'].fillna(train_fill_in)

# Updated Optimization for Age
new_age_bins = [0, 12, 18, 30, 60, 80, 100]
new_age_labels = [0, 1, 2, 3, 4, 5]
train_data['AgeCategories'] = pd.cut(train_data['Age'], bins=new_age_bins, labels=new_age_labels)
train_data['AgeCategories'] = train_data['AgeCategories'].astype(int)

# Updated Optimization for Family sizes and checking to see if bigger fams survive more
train_data['FamilySize'] = (train_data['SibSp'] + (train_data['Parch'] + 1))

# Updated Optimization seeing if solo chances of survival 
train_data['Solo'] = (train_data['FamilySize'] == 1).astype(int)
#Adding and seeeing fare per person
train_data['FarePer'] = train_data['Fare'] / train_data['FamilySize']

# Fill missing Embarked values with most common port ('S')
train_data['Embarked'] = train_data['Embarked'].fillna('S')

# Encode Sex (male=0, female=1)
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1}).fillna(0)

# fill missing Fare values if needed
train_fill_in_fare = train_data['Fare'].median()
train_data['Fare'] = train_data['Fare'].fillna(train_fill_in_fare)

# One-hot encode Embarked
train_data = pd.get_dummies(train_data, columns=['Embarked'], prefix='Embarked')

# Ensure all three dummy columns exist
for col in ['Embarked_C', 'Embarked_Q', 'Embarked_S']:
    if col not in train_data.columns:
        train_data[col] = 0

# Impute missing values in test_cleaned
imputer = SimpleImputer(strategy='mean')
test_cleaned_imputed = pd.DataFrame(imputer.fit_transform(test_cleaned), columns=test_cleaned.columns)

#Binary Classification Model Implementation
#Referencing: MN Lab 4 Neural Network Implementation 

#splitting into x(features) and y(target) + training and testing sets
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'AgeCategories', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'FamilySize', 'Solo', 'FarePer']
X = train_data[features] #every col but 'survived'
y = train_data ['Survived']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2 ,random_state=42) #42 because that's what we use in class

#fitting the model and evaluating
rf = RandomForestClassifier(
    n_estimators = 300,
    max_depth = 7,
    min_samples_split= 4,
    min_samples_leaf= 2,
    max_features='sqrt',
    random_state=42
)
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)


#prediction on test set
test_pred = rf.predict(test_cleaned_imputed[features])
#final output
Survived = pd.DataFrame({ 
'PassengerId': test_data['PassengerId'],
'Survived': test_pred })
print(Survived)
Survived.to_csv('submission_mn_two.csv', index=False)

0.8268156424581006
     PassengerId  Survived
0            892         0
1            893         0
2            894         0
3            895         0
4            896         0
..           ...       ...
413         1305         0
414         1306         1
415         1307         0
416         1308         0
417         1309         0

[418 rows x 2 columns]


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e20c63f8-5329-4013-bde2-b241ff10039b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>